In [1]:
from sentence_transformers import SentenceTransformer
from sentence_transformers.util import cos_sim

model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

import pandas as pd

import matplotlib.pyplot as plt

from scipy import stats

from scipy.stats import shapiro, levene

import scikit_posthocs as sp

from scipy.stats import kruskal

c:\Users\marco\Tesi_2025-26-La-Geometria-degli-Embedding-di-un-LLm\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4082.95it/s]


In [2]:
# carico il dataset
taxonomic_tree = pd.read_csv("taxonomic_pairs.csv")

In [4]:
#creo set all_words con tutte le parole per poter fare embedding
all_words = set()

# sinonimi
all_words.update(taxonomic_tree["word1"])
all_words.update(taxonomic_tree["word2"])

print(all_words)
print(len(all_words))


{'boat', 'vehicle', 'hovercraft', 'vessel', 'steamroller', 'lander', 'craft', 'missile', 'luge', 'aircraft', 'ship', 'yacht', 'spacecraft', 'toboggan', 'galley', 'sled', 'bobsled', 'sidewinder', 'rocket', 'starship'}
20


In [5]:
#calcolo gli embedding una sola volta grazie a all_words, li metto in un dizionario
embeddings = {}

for word in all_words:
    
    embeddings[word] = model.encode(
        word,
        convert_to_tensor=True
    )


print("Embedding creati:", len(embeddings))

Embedding creati: 20


In [6]:
#salvo il dizionario così da non doverli calcolare ogni volta che apro il progetto

import pickle

with open("embeddings.pkl_2", "wb") as f:
    pickle.dump(embeddings, f)

In [8]:
#codice da eseguire per avere gli embedding

import pickle

with open("embeddings.pkl_2", "rb") as f:
    embeddings = pickle.load(f)

In [9]:
def add_cosine_similarity(df, col1, col2):
    #in input entrano: il database delle coppie, la colonna della parola e la colonna della parola associata
    #si crea una lista  per i risultati della cosine similarity
    #si calcola la cosine similarity usando gli embedding già calcolati
    #si crea una colonna nel dataset di partenza con i risultati della cosine similarity
    
    similarities = []

    for w1, w2 in zip(df[col1], df[col2]):

        emb1 = embeddings[w1]
        emb2 = embeddings[w2]

        similarity = cos_sim(
            emb1,
            emb2
        ).item()

        similarities.append(similarity)


    df["cosine_similarity"] = similarities

    return df

In [10]:
#funzione applicata all'albero

taxonomic_df = add_cosine_similarity(
    taxonomic_tree,
    "word1",
    "word2"
)

print(taxonomic_df.head())

     word1       word2  taxonomic_distance  cosine_similarity
0  vehicle      rocket                   1           0.417660
1  vehicle     missile                   2           0.399021
2  vehicle  sidewinder                   3           0.230047
3  vehicle       craft                   1           0.319384
4  vehicle    aircraft                   2           0.516191


In [11]:
#salvo i risultati
taxonomic_df.to_csv(
    "results_taxonomic_tree_similarity.csv",
    index=False
)


In [13]:
#per caricare dataset con cosine similarity

taxonomic_df = pd.read_csv("results_taxonomic_tree_similarity.csv")